# E-Commerce Data Warehouse — Exploration Notebook

This notebook provides interactive exploration of the data at each pipeline stage.

**Pipeline stages:**
1. Raw staging data overview
2. Transformation validation
3. DW dimension & fact table exploration
4. Visualisations


In [ ]:
import json, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2

# Styling
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

# Load DB config
with open('../config/db_config.json') as f:
    cfg = json.load(f)

conn = psycopg2.connect(
    host=cfg['host'], port=cfg['port'],
    dbname=cfg['database'], user=cfg['user'], password=cfg['password']
)
print('Connected to PostgreSQL ✓')

## 1. Staging Data Overview

In [ ]:
tables = ['countries','currencies','customers','products','orders','order_items']
for t in tables:
    df = pd.read_sql(f'SELECT COUNT(*) AS cnt FROM staging.{t}', conn)
    print(f'  staging.{t:15s} → {df.iloc[0,0]:,} rows')

## 2. DW Table Counts

In [ ]:
dw_tables = ['dim_date','dim_location','dim_customer','dim_product','dim_payment','fact_sales','fact_orders']
for t in dw_tables:
    df = pd.read_sql(f'SELECT COUNT(*) AS cnt FROM dw.{t}', conn)
    print(f'  dw.{t:20s} → {df.iloc[0,0]:,} rows')

## 3. Monthly Revenue Chart

In [ ]:
df = pd.read_sql("""
    SELECT d.year, d.month_number, d.month_short || ' ' || d.year AS period,
           SUM(fo.net_revenue) AS net_revenue
    FROM dw.fact_orders fo
    JOIN dw.dim_date d ON fo.order_date_key = d.date_key
    WHERE fo.order_status NOT IN ('Cancelled','Returned')
    GROUP BY d.year, d.month_number, d.month_short
    ORDER BY d.year, d.month_number
""", conn)

fig, ax = plt.subplots(figsize=(16,5))
ax.bar(range(len(df)), df['net_revenue'], color=sns.color_palette('husl', len(df)))
ax.set_xticks(range(len(df)))
ax.set_xticklabels(df['period'], rotation=45, ha='right', fontsize=8)
ax.set_title('Monthly Net Revenue', fontsize=14, fontweight='bold')
ax.set_ylabel('Net Revenue (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## 4. Revenue by Category

In [ ]:
df = pd.read_sql("""
    SELECT dp.category, SUM(fs.line_net) AS net_revenue,
           ROUND(AVG(fs.margin_pct)::numeric,1) AS avg_margin
    FROM dw.fact_sales fs
    JOIN dw.dim_product dp ON fs.product_key = dp.product_key
    WHERE fs.order_status NOT IN ('Cancelled','Returned')
    GROUP BY dp.category ORDER BY net_revenue DESC
""", conn)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(df['category'], df['net_revenue'], color='steelblue')
axes[0].set_title('Net Revenue by Category')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
axes[1].barh(df['category'], df['avg_margin'], color='coral')
axes[1].set_title('Avg Margin % by Category')
axes[1].set_xlabel('Margin %')
plt.tight_layout(); plt.show()

## 5. Customer Loyalty Tier Analysis

In [ ]:
df = pd.read_sql("""
    SELECT dc.loyalty_tier, COUNT(DISTINCT dc.customer_key) AS customers,
           ROUND(SUM(fo.net_revenue)::numeric,2) AS total_revenue
    FROM dw.dim_customer dc
    LEFT JOIN dw.fact_orders fo ON dc.customer_key = fo.customer_key
        AND fo.order_status NOT IN ('Cancelled','Returned')
    GROUP BY dc.loyalty_tier ORDER BY total_revenue DESC NULLS LAST
""", conn)
print(df.to_string(index=False))

In [ ]:
conn.close()
print('Connection closed.')